# Tópicos de Big Data — Segundo examen parcial
## Examen práctico · Versión A

**Tiempo máximo:** 90 minutos  
**Calificación de este examen práctico:** 100 puntos (escala de 0 a 100)  
**Nombre:** Victor Alejandro Chávez Valadez  
**Grupo:** 512  **Fecha:** 26/10/2026

### Situación de análisis
Se dispone de un conjunto de datos de viajes de taxi de Nueva York. Tu tarea consiste en aplicar **K-Means** para responder la siguiente pregunta:

> **¿Qué tipos de viajes pagados en efectivo se pueden identificar según la distancia recorrida y la duración del viaje?**

Trabajarás únicamente con los viajes cuyo tipo de pago sea **`Cash`** y utilizarás las variables **`trip_distance`** y **`trip_duration`**.

### Reglas de entrega
- Escribe tu propio código en las celdas vacías que aparecen después de cada instrucción.
- Ejecuta todas las celdas en orden y conserva las tablas y gráficas resultantes.
- Utiliza **Python, Pandas, NumPy, Matplotlib y scikit-learn**.
- No es necesario utilizar técnicas distintas de las estudiadas en clase.
- Guarda el archivo como **`Examen_A_Apellido_Nombre.ipynb`** y entrégalo con sus resultados visibles.
- El número de clústeres **no está predeterminado**: deberás seleccionarlo y justificarlo con evidencia.

**Ubicación obligatoria del dataset:** dentro de la carpeta de tu examen, guarda el archivo `nyc_taxi_examen.csv` en la subcarpeta `data/raw/`. La ruta relativa desde este notebook será **`data/raw/nyc_taxi_examen.csv`** (no es una ruta absoluta del sistema).

**Estructura de carpetas:**
```text
Examen_A/
└── notebooks/
    └── Examen_A_Apellido_Nombre.ipynb
└── data/
    └── raw/
        └── nyc_taxi_examen.csv
```

## 1. Carga y exploración de datos — 7 puntos · 10 minutos

**Escribe el código necesario para:**
1. Importar las bibliotecas que utilizarás.
2. Cargar el archivo CSV **`../data/raw/nyc_taxi_examen.csv`** en un DataFrame llamado `df`. La ruta es relativa a la carpeta donde está este notebook.
3. Mostrar las **primeras cinco filas**, las **dimensiones**, los **nombres de las columnas** y los **tipos de datos**.
4. Mostrar un resumen estadístico de las columnas numéricas.
5. Comprobar qué valores existen en la columna `payment_type`.

**Evidencia:** deja visibles los resultados de exploración.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ruta = "../data/raw/nyc_taxi_examen.csv"
df = pd.read_csv(ruta, index_col=0)   # la primera columna es el índice guardado del CSV

print("Primeras cinco filas:")
display(df.head())
print("Dimensiones:", df.shape)
print("\nColumnas:", list(df.columns))
print("\nTipos de datos:")
print(df.dtypes)
print("\nResumen estadístico de columnas numéricas:")
display(df.describe())
print("Valores en payment_type:")
print(df["payment_type"].value_counts(dropna=False))

## 2. Revisión y limpieza de datos — 14 puntos · 15 minutos

**Escribe el código necesario para:**
1. Crear una copia de los datos originales llamada `df_limpio`.
2. Contar los **valores nulos por columna** y los **registros duplicados** antes de la limpieza.
3. Eliminar registros duplicados y filas que contengan valores nulos.
4. Conservar únicamente los registros que cumplan **todas** estas condiciones:
   - `trip_distance > 0`
   - `passenger_count > 0` y `passenger_count < 100`
   - `fare_amount > 0` y `fare_amount < 10000`
   - `trip_duration > 0`
5. Mostrar cuántos registros quedan después de limpiar.

**Importante:** no elimines registros por valores atípicos utilizando percentiles ni otros métodos adicionales. Si el archivo ya fue depurado, verifica igualmente estas condiciones.

**Evidencia:** muestra los conteos y las dimensiones antes y después de limpiar.

In [ ]:
df_limpio = df.copy()
print("Dimensiones antes de limpiar:", df_limpio.shape)
print("\nValores nulos por columna:")
print(df_limpio.isnull().sum())
print("\nRegistros duplicados:", df_limpio.duplicated().sum())

df_limpio = df_limpio.drop_duplicates()
df_limpio = df_limpio.dropna()

df_limpio = df_limpio[
    (df_limpio["trip_distance"] > 0) &
    (df_limpio["passenger_count"] > 0) & (df_limpio["passenger_count"] < 100) &
    (df_limpio["fare_amount"] > 0) & (df_limpio["fare_amount"] < 10000) &
    (df_limpio["trip_duration"] > 0)
]

print("\nDimensiones después de limpiar:", df_limpio.shape)
print("Registros restantes:", len(df_limpio))
print("Registros eliminados:", len(df) - len(df_limpio))

## 3. Filtrado y selección de variables — 7 puntos · 5 minutos

**Escribe el código necesario para:**
1. Filtrar los registros de `df_limpio` donde `payment_type` sea exactamente **`Cash`**. Guarda el resultado en `df_cash`.
2. Seleccionar únicamente **`trip_distance`** y **`trip_duration`** para el agrupamiento. Guarda esas dos columnas en un DataFrame llamado `X`.
3. Mostrar la cantidad de viajes filtrados y las estadísticas descriptivas de `X`.

**Recuerda:** utiliza **únicamente estas dos variables** para entrenar K-Means.

**Evidencia:** cantidad de registros y tabla descriptiva.

In [ ]:
df_cash = df_limpio[df_limpio["payment_type"] == "Cash"].copy()
X = df_cash[["trip_distance", "trip_duration"]]

print("Cantidad de viajes pagados en efectivo:", len(df_cash))
display(X.describe())

## 4. Transformación y estandarización — 14 puntos · 10 minutos

**Escribe el código necesario para:**
1. Aplicar `np.log1p()` a las dos variables de `X`. Guarda el resultado en `X_log`.
2. Crear un objeto `StandardScaler` y ajustar/transformar `X_log`.
3. Guardar los datos estandarizados en `X_escalado`.
4. Mostrar un resumen estadístico de las variables transformadas y estandarizadas para comprobar el resultado.

**Evidencia:** tabla con media, desviación estándar y otros estadísticos de las variables estandarizadas.

> La transformación logarítmica y la estandarización se aplican para el modelo; conserva `df_cash` con las unidades originales para interpretar los grupos al final.

In [ ]:
X_log = np.log1p(X)

scaler = StandardScaler()
X_escalado = scaler.fit_transform(X_log)

X_escalado_df = pd.DataFrame(X_escalado, columns=X.columns, index=X.index)
print("Variables con log1p:")
display(X_log.describe())
print("Variables con log1p + StandardScaler (media ≈ 0, desviación ≈ 1):")
display(X_escalado_df.describe().round(4))

## 5. Evaluación del número de clústeres — 21 puntos · 20 minutos

**Escribe el código necesario para:**
1. Evaluar los valores de **`k = 2, 3, 4, 5 y 6`** mediante un ciclo.
2. Para cada valor de `k`, entrenar `KMeans` con `random_state=42` y `n_init=10`.
3. Calcular y almacenar la **inercia** (`inertia_`).
4. Calcular y almacenar el **coeficiente de silueta** con `silhouette_score`, usando `sample_size=5000` y `random_state=42` para que el cálculo sea manejable.
5. Construir y mostrar una **tabla comparativa** con las columnas `k`, `Inercia` y `Silueta`.
6. Generar **dos gráficas separadas**: una para el método del codo y otra para el coeficiente de silueta. Ambas deben tener título y etiquetas en los ejes.

**Evidencia:** tabla y dos gráficas.

**Nota:** una inercia menor no basta por sí sola para elegir `k`; considera también la silueta y la interpretación de los grupos.

In [ ]:
valores_k = [2, 3, 4, 5, 6]
inercias = []
siluetas = []

for k in valores_k:
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas = modelo.fit_predict(X_escalado)
    inercias.append(modelo.inertia_)
    siluetas.append(silhouette_score(X_escalado, etiquetas, sample_size=5000, random_state=42))

tabla_metricas = pd.DataFrame({"k": valores_k, "Inercia": inercias, "Silueta": siluetas})
display(tabla_metricas.round(4))

plt.figure(figsize=(7, 4))
plt.plot(valores_k, inercias, marker="o")
plt.title("Método del codo")
plt.xlabel("Número de clústeres (k)")
plt.ylabel("Inercia")
plt.xticks(valores_k)
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(valores_k, siluetas, marker="o", color="darkorange")
plt.title("Coeficiente de silueta")
plt.xlabel("Número de clústeres (k)")
plt.ylabel("Silueta")
plt.xticks(valores_k)
plt.grid(True)
plt.show()

## 6. Modelo final y visualización — 14 puntos · 15 minutos

**Escribe el código necesario para:**
1. Elegir un valor de `k` a partir de los resultados anteriores.
2. Entrenar un modelo final `KMeans` con ese valor, `random_state=42` y `n_init=10`.
3. Obtener las etiquetas de clúster para cada viaje.
4. Generar una **gráfica de dispersión** usando las dos columnas de `X_escalado`: eje X = **`trip_distance`**, eje Y = **`trip_duration`**, ambas transformadas y estandarizadas.
5. Colorear los puntos según el clúster asignado.
6. Dibujar los **centroides** del modelo con marcadores **X** de tamaño visible y un color que contraste.
7. Agregar título, nombres de ejes y leyenda de centroides.

**Evidencia:** gráfica final con clústeres y centroides.

**Nota sobre la visualización:** la gráfica debe mostrar los datos y centroides **transformados y estandarizados**, tal como los utiliza K-Means. Los valores de sus ejes no representan directamente las unidades originales.

In [ ]:
k_final = 3

modelo_final = KMeans(n_clusters=k_final, random_state=42, n_init=10)
etiquetas = modelo_final.fit_predict(X_escalado)
centroides = modelo_final.cluster_centers_

plt.figure(figsize=(9, 6))
dispersion = plt.scatter(X_escalado[:, 0], X_escalado[:, 1], c=etiquetas, cmap="viridis", s=3, alpha=0.4)
plt.scatter(centroides[:, 0], centroides[:, 1], marker="X", s=300, c="red",
            edgecolors="black", label="Centroides")
plt.title(f"Clústeres K-Means (k = {k_final}) — viajes pagados en efectivo")
plt.xlabel("trip_distance (log1p + estandarizada)")
plt.ylabel("trip_duration (log1p + estandarizada)")
plt.colorbar(dispersion, label="Clúster")
plt.legend()
plt.show()

## 7. Estadísticos de los clústeres y preguntas de análisis — 23 puntos · 15 minutos

**Parte A. Tabla de estadísticos — 11 puntos**

**Importante:** calcula los estadísticos descriptivos usando los **datos originales de las variables, sin aplicarles la transformación logarítmica ni la estandarización**. Usa las etiquetas del modelo para agrupar los registros, pero toma los valores de `df_cash`, no de `X_escalado`.

**Escribe el código necesario para:**
1. Crear un DataFrame llamado `df_resultado` a partir de `df_cash`.
2. Agregar una columna `cluster` con las etiquetas obtenidas en el modelo final.
3. Agrupar por `cluster` y calcular, para cada grupo:
   - **Cantidad de viajes**.
   - **Media y mediana** de `trip_distance`.
   - **Media y mediana** de `trip_duration`.
4. Mostrar la tabla con valores redondeados a dos decimales. Los resultados deben expresarse en sus **unidades originales**: millas para la distancia y minutos para la duración.

**Evidencia:** tabla de estadísticos por clúster.

In [ ]:
df_resultado = df_cash.copy()
df_resultado["cluster"] = etiquetas

tabla_clusters = df_resultado.groupby("cluster").agg(
    cantidad_viajes=("trip_distance", "size"),
    distancia_media_millas=("trip_distance", "mean"),
    distancia_mediana_millas=("trip_distance", "median"),
    duracion_media_min=("trip_duration", "mean"),
    duracion_mediana_min=("trip_duration", "median"),
).round(2)

display(tabla_clusters)

**Parte B. Preguntas de análisis — 12 puntos (3 puntos por pregunta).** Responde las cuatro preguntas con tus propias palabras. Usa las métricas, la gráfica y la tabla de estadísticos como evidencia. No basta con mencionar el número de clúster.

**Pregunta 1 — Selección de k.** ¿Qué valor de `k` elegiste y por qué? Justifica tu decisión utilizando **inercia, silueta** y, si corresponde, la claridad de los grupos obtenidos.

**Respuesta:**
Elegí **k = 3**. En el método del codo, la inercia baja de 227,681 (k=2) a 137,512 (k=3), una reducción de aproximadamente 40 %; a partir de ahí las mejoras son cada vez menores (≈27 % de k=3 a k=4 y ≈21 % de k=4 a k=5), por lo que el "codo" se ubica en k = 3. En cuanto a la silueta, k = 2 obtiene el valor más alto (0.532), pero k = 3 mantiene una silueta aceptable (0.448), claramente superior a la de k = 4, 5 y 6 (0.419, 0.389 y 0.374). Además, k = 2 solo separa viajes "cortos" y "largos", mientras que k = 3 produce tres grupos bien diferenciados y fáciles de interpretar (cortos, medianos y largos), como se observa en la gráfica de dispersión. Por eso k = 3 ofrece el mejor equilibrio entre compacidad, separación e interpretabilidad.

**Pregunta 2 — Interpretación de grupos.** Describe las características de **cada clúster** a partir de la cantidad de viajes, las medias y las medianas. Asigna a cada grupo un nombre descriptivo.

**Respuesta:**
- **Clúster 2 — "Viajes muy cortos / de cuadra" (94,169 viajes):** distancia media de 0.80 millas (mediana 0.76) y duración media de 5.72 min (mediana 5.82). Son trayectos de menos de una milla y de pocos minutos, típicos de traslados dentro del mismo vecindario.
- **Clúster 1 — "Viajes urbanos medianos" (125,521 viajes):** distancia media de 2.30 millas (mediana 1.94) y duración media de 15.43 min (mediana 14.00). Son los recorridos cotidianos dentro de la ciudad, entre zonas cercanas.
- **Clúster 0 — "Viajes largos / interzona o aeropuerto" (62,395 viajes):** distancia media de 11.55 millas (mediana 9.81) y duración media de 39.48 min (mediana 31.40). Son trayectos largos, por ejemplo hacia aeropuertos u otros distritos. En este grupo la media es bastante mayor que la mediana, lo que indica que contiene algunos viajes extremos (muy largos o muy lentos) que elevan el promedio.

**Pregunta 3 — Análisis sencillo.** ¿Qué clúster presenta la mayor duración promedio? ¿También tiene la mayor distancia promedio? Explica usando la tabla.

**Respuesta:**
El **clúster 0 ("Viajes largos")** tiene la mayor duración promedio, con 39.48 minutos, y **también** tiene la mayor distancia promedio, con 11.55 millas. Esto es coherente, ya que la distancia y la duración están relacionadas: los viajes que recorren más millas suelen tardar más tiempo. Su duración promedio es aproximadamente 2.6 veces la del clúster 1 (15.43 min) y casi 7 veces la del clúster 2 (5.72 min).

**Pregunta 4 — Análisis sencillo.** ¿Cuál es el grupo más numeroso? ¿Qué características de distancia y duración predominan en ese grupo?

**Respuesta:**
El grupo más numeroso es el **clúster 1 ("Viajes urbanos medianos")**, con 125,521 viajes, aproximadamente el 44.5 % de los 282,085 viajes pagados en efectivo. En este grupo predominan distancias cortas o moderadas de alrededor de 2 millas (media de 2.30 y mediana de 1.94) y duraciones de entre 14 y 15 minutos (media de 15.43 y mediana de 14.00). Como la media y la mediana son similares, el grupo es bastante homogéneo. Esto indica que el viaje típico pagado en efectivo es un recorrido urbano corto-mediano, de un par de millas y alrededor de un cuarto de hora.

---
### Lista de verificación antes de entregar

- [x] Se muestran la carga, exploración, limpieza y filtrado.
- [x] Se utilizaron las **dos variables indicadas**.
- [x] Se aplicaron `log1p` y `StandardScaler`.
- [x] Se calcularon inercia y silueta para `k = 2...6`.
- [x] Se muestran la tabla de métricas y las dos gráficas de evaluación.
- [x] Se visualizan los clústeres y sus centroides.
- [x] Se presenta la tabla de estadísticos calculados sobre los **datos originales**, sin transformación ni estandarización, en sus unidades correspondientes.
- [x] Se respondieron las cuatro preguntas finales.

**Distribución de puntos (sobre 100):** carga y exploración: 7; limpieza: 14; filtrado y selección: 7; transformación y estandarización: 14; evaluación de k: 21; modelo y visualización: 14; estadísticos: 11; preguntas de análisis: 12. **Total: 100 puntos.**